# Generate Fact Data

This notebook generates transaction data (Facts): Orders and Invoices, ensuring referential integrity with the generated dimensions.

In [ ]:
import pandas as pd
import numpy as np
import glob
import sqlite3
import random
from datetime import datetime, timedelta

%run 00_common_utils.ipynb

## Load Latest Dimensions

In [ ]:
def load_latest_snapshot(prefix):
    files = glob.glob(str(WAREHOUSE_DIR / f'{prefix}_*.csv'))
    if not files:
        raise FileNotFoundError(f"No snapshot found for {prefix}")
    latest_file = max(files, key=os.path.getctime)
    logger.info(f"Loading {latest_file}")
    return pd.read_csv(latest_file)

try:
    dim_products = load_latest_snapshot('dim_products')
    dim_customers = load_latest_snapshot('dim_customers')
    dim_employees = load_latest_snapshot('dim_employees')
except Exception as e:
    logger.error(f"Failed to load dimensions: {e}")
    raise

## Generation Logic

In [ ]:
NUM_DAYS = 30
TRANSACTIONS_PER_DAY_RANGE = (50, 150)

customers_list = dim_customers['customer_code'].tolist()
employees_list = dim_employees['employee_code'].tolist()
products_df = dim_products.set_index('product_code')
products_list = products_df.index.tolist()

fact_invoices = []
fact_invoice_lines = []

start_date = datetime.now() - timedelta(days=NUM_DAYS)
invoice_counter = 1

for day in range(NUM_DAYS):
    current_date = start_date + timedelta(days=day)
    num_tx = random.randint(*TRANSACTIONS_PER_DAY_RANGE)
    
    for _ in range(num_tx):
        invoice_code = f'HDIP{invoice_counter:05d}'
        customer = random.choice(customers_list)
        employee = random.choice(employees_list)
        time = current_date + timedelta(hours=random.randint(8, 20), minutes=random.randint(0, 59))
        
        # Generate lines
        num_lines = random.randint(1, 5)
        invoice_total = 0
        
        # Select random distinct products for this invoice
        selected_products = random.sample(products_list, num_lines)
        for prod_code in selected_products:
            qty = random.randint(1, 10)
            unit_price = products_df.loc[prod_code, 'sale_price']
            line_total = qty * unit_price
            invoice_total += line_total
            
            fact_invoice_lines.append({
                'invoice_code': invoice_code,
                'product_code': prod_code,
                'quantity': qty,
                'unit_price': unit_price,
                'line_total': line_total
            })
        
        fact_invoices.append({
            'invoice_code': invoice_code,
            'customer_code': customer,
            'employee_code': employee,
            'timestamp': time,
            'total_amount': invoice_total,
            'payment_method': random.choice(['Tiền mặt', 'Chuyển khoản', 'Thẻ'])
        })
        
        invoice_counter += 1

df_invoices = pd.DataFrame(fact_invoices)
df_invoice_lines = pd.DataFrame(fact_invoice_lines)

logger.info(f"Generated {len(df_invoices)} invoices and {len(df_invoice_lines)} invoice lines.")

## Export Data

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save to warehouse as CSV
df_invoices.to_csv(WAREHOUSE_DIR / f'fact_invoices_{timestamp}.csv', index=False)
df_invoice_lines.to_csv(WAREHOUSE_DIR / f'fact_invoice_lines_{timestamp}.csv', index=False)

# Save to SQLite
db_path = WAREHOUSE_DIR / 'retail_synthetic.db'
conn = sqlite3.connect(db_path)

dim_products.to_sql('DIM_PRODUCTS', conn, if_exists='replace', index=False)
dim_customers.to_sql('DIM_CUSTOMERS', conn, if_exists='replace', index=False)
dim_employees.to_sql('DIM_EMPLOYEES', conn, if_exists='replace', index=False)
df_invoices.to_sql('FACT_INVOICES', conn, if_exists='replace', index=False)
df_invoice_lines.to_sql('FACT_INVOICES_LINES', conn, if_exists='replace', index=False)
conn.close()

logger.info(f"Data saved to SQLite database at {db_path}")